In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report

df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nMissing %:")
print((df.isnull().sum() / len(df) * 100).round(1))

Shape: (891, 12)

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Missing %:
PassengerId     0.0
Survived        0.0
Pclass          0.0
Name            0.0
Sex             0.0
Age            19.9
SibSp           0.0
Parch           0.0
Ticket          0.0
Fare            0.0
Cabin          77.1
Embarked        0.2
dtype: float64


In [11]:
# Look at each column with missing values:
# Age    → 19.9% missing → FILL IT (too much to drop)
# Cabin  → 77.1% missing → DROP IT (too much missing to be useful)
# Embarked → 0.2% missing → FILL with mode (only 2 rows)

# 3 strategies for filling:

# Strategy 1 — Mean/Median (for numbers)
print("Age median:", df['Age'].median())
print("Age mean:", df['Age'].mean())
# Use MEDIAN when data is skewed, MEAN when it's normal
# Age has outliers → use median

# Strategy 2 — Mode (for categories)
print("Embarked mode:", df['Embarked'].mode()[0])

# Strategy 3 — KNN Imputer (smarter — uses similar passengers)
# We'll use this inside the Pipeline below

Age median: 28.0
Age mean: 29.69911764705882
Embarked mode: S


In [12]:
# LabelEncoder → converts to numbers: male=1, female=0
# USE ONLY for binary categories (2 values) or tree-based models
le = LabelEncoder()
print(le.fit_transform(['male','female','male','male']))
# Output: [1, 0, 1, 1]

# OneHotEncoder → creates separate 0/1 column for each category
# USE for categories with 3+ values with any model
# Example: Embarked has S, C, Q → becomes 3 columns
#   Embarked_S  Embarked_C  Embarked_Q
#       1           0           0       ← S
#       0           1           0       ← C
#       0           0           1       ← Q

# Why not LabelEncoder for Embarked?
# LabelEncoder gives S=2, C=0, Q=1
# Model thinks S > Q > C mathematically — WRONG!
# OneHotEncoder has no such ordering problem

[1 0 1 1]


In [13]:
# First — feature engineering (same as Day 12)
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
df['Title'] = df['Title'].replace(
    ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
df['Title'] = df['Title'].replace({'Mlle':'Miss', 'Ms':'Miss', 'Mme':'Mrs'})
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
df['HasCabin'] = df['Cabin'].notnull().astype(int)
df['Embarked'] = df['Embarked'].fillna('S')

# Define which columns are numeric and which are categorical
numeric_features = ['Age', 'Fare', 'FamilySize', 'IsAlone', 'HasCabin', 'Pclass']
categorical_features = ['Sex', 'Embarked', 'Title']

X = df[numeric_features + categorical_features]
y = df['Survived']

# Split FIRST — before any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (712, 9)
Test size: (179, 9)


In [14]:
# Limit the tree depth so the model doesn't memorize
better_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        max_depth=6,        # don't grow too deep
        min_samples_leaf=4, # each leaf needs at least 4 samples
        random_state=42
    ))
])

better_pipeline.fit(X_train, y_train)
print(f"Train: {better_pipeline.score(X_train, y_train):.4f}")
print(f"Test:  {better_pipeline.score(X_test, y_test):.4f}")

Train: 0.8610
Test:  0.8212


In [15]:
y_pred = full_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.86      0.85       105
           1       0.79      0.78      0.79        74

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179



In [16]:
# KNN Imputer fills missing Age by looking at
# the K most similar passengers and averaging their age
# Much smarter than just using the median!

numeric_pipeline_knn = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])

preprocessor_knn = ColumnTransformer([
    ('num', numeric_pipeline_knn, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

pipeline_knn = Pipeline([
    ('preprocessor', preprocessor_knn),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline_knn.fit(X_train, y_train)
knn_score = pipeline_knn.score(X_test, y_test)
print(f"With KNN Imputer: {knn_score:.4f}")
print(f"With Simple Imputer: {test_score:.4f}")

With KNN Imputer: 0.8268
With Simple Imputer: 0.8268


In [17]:

# This is the real test — can your pipeline handle new data?
new_passenger = pd.DataFrame({
    'Age': [None],          # missing age!
    'Fare': [52.0],
    'FamilySize': [1],
    'IsAlone': [1],
    'HasCabin': [0],
    'Pclass': [2],
    'Sex': ['male'],
    'Embarked': ['S'],
    'Title': ['Mr']
})

prediction = full_pipeline.predict(new_passenger)
probability = full_pipeline.predict_proba(new_passenger)

print(f"Survived: {'Yes' if prediction[0]==1 else 'No'}")
print(f"Survival probability: {probability[0][1]:.2%}")

Survived: No
Survival probability: 15.60%
